In [ ]:
print(1)

In [ ]:

import json
import glob
import logging
import warnings
from collections import deque
from datetime import datetime
from pathlib import Path
from typing import Optional, Tuple, Dict, List

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
logger = logging.getLogger(__name__)

# ── Paths (adjust to your layout) ────────────────────────────────────────────
ROOT_DIR    = Path("/Users/darshan/college/6th_sem/Minor")         # repo root
LOG_ROOT    = ROOT_DIR / "Logs"              # contains logs1/, logs_mesh1/ …
MODEL_DIR   = ROOT_DIR / "Models"
DATASET_DIR = ROOT_DIR / "Datasets"
MODEL_DIR.mkdir(exist_ok=True)
DATASET_DIR.mkdir(exist_ok=True)

# ── Feature column order (must stay the same everywhere) ─────────────────────
FEATURE_COLS = [
    "pit_size",
    "pit_growth_rate",
    "cs_size",
    "cache_hit_ratio",       # ← now delta-based
    "satisfaction_ratio",    # ← now delta-based
    "unsatisfied_ratio",     # ← now delta-based
    "in_interests_rate",
    "out_interests_rate",
    "in_data_rate",
    "nack_rate",
]

RATE_COLS = [
    "pit_growth_rate", "in_interests_rate", "out_interests_rate",
    "in_data_rate", "nack_rate",
]

REQUIRED_RAW_COLS = [
    "timestamp", "node",
    "nPitEntries", "nInInterests", "nOutInterests",
    "nInData", "nInNacks", "nOutNacks",
    "nSatisfiedInterests", "nUnsatisfiedInterests",
    "nCsEntries", "nHits", "nMisses",
]

# ── Training hyper-parameters ─────────────────────────────────────────────────
WARMUP_SECONDS      = 600    # drop first 10 min per node (startup noise)
IF_N_ESTIMATORS     = 200
IF_CONTAMINATION    = 0.01   # 1 % anomaly rate assumed in training data
IF_RANDOM_STATE     = 42
ANOMALY_PERCENTILE  = 1      # threshold = bottom-1 % of training scores
CLIP_QUANTILE_LO    = 0.01
CLIP_QUANTILE_HI    = 0.99

# ── Artifact file names ────────────────────────────────────────────────────────
MODEL_PATH      = MODEL_DIR / "ndn_isolation_forest.joblib"
THRESHOLD_PATH  = MODEL_DIR / "ndn_threshold.json"
FEATURES_PATH   = MODEL_DIR / "feature_cols.json"
CLIP_BOUNDS_PATH= MODEL_DIR / "clip_bounds.json"   # ← NEW: saved clip bounds

In [ ]:

def parse_jsonl_file(filepath: Path) -> pd.DataFrame:
    """Parse a single *.jsonl file into a DataFrame."""
    records = []
    with open(filepath) as fh:
        for lineno, line in enumerate(fh, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                logger.warning(f"  Bad JSON at {filepath.name}:{lineno} — skipped")
    return pd.DataFrame(records) if records else pd.DataFrame()


def load_raw_logs(scenarios: List[str], log_root: Path = LOG_ROOT) -> pd.DataFrame:
    """
    Load all *_metrics.jsonl files from the given scenario directories.

    Args:
        scenarios : list of subdirectory names under log_root
                    e.g. ["logs1", "logs_mesh1", "logs_dumbbell1"]
        log_root  : root directory that contains the scenario folders

    Returns:
        DataFrame with one row per raw log entry + provenance columns
        (_scenario, _source_file).
    """
    metric_subdirs = ["normal_traffic_metrics", "cache_pollution_metrics"]
    found = []

    for scenario in scenarios:
        base = log_root / scenario
        for sub in metric_subdirs:
            pattern = str(base / "**" / sub / "*_metrics.jsonl")
            found.extend(glob.glob(pattern, recursive=True))

    if not found:
        raise FileNotFoundError(
            f"No *_metrics.jsonl files found under {log_root} "
            f"for scenarios: {scenarios}"
        )

    logger.info(f"Found {len(found)} JSONL files across {len(scenarios)} scenario(s)")

    frames = []
    for fp in tqdm(sorted(set(found)), desc="Parsing logs"):
        p  = Path(fp)
        df = parse_jsonl_file(p)
        if df.empty:
            continue
        # Node name lives in the directory just above the metrics sub-folder
        # e.g.  logs1 / c1 / normal_traffic_metrics / c1_metrics.jsonl
        #                ^^  p.parent.parent.name
        node_from_path = p.parent.parent.name
        if "node" not in df.columns or df["node"].isna().all():
            df["node"] = node_from_path
        df["_scenario"]    = p.parts[-4]   # scenario folder name
        df["_source_file"] = fp
        frames.append(df)

    if not frames:
        raise ValueError("All log files were empty or unreadable.")

    combined = pd.concat(frames, ignore_index=True)
    logger.info(f"Raw records loaded: {len(combined):,}")
    return combined


def load_single_jsonl(filepath: Path) -> pd.DataFrame:
    """
    Convenience wrapper: load a single JSONL file (e.g. c1_after_600.jsonl)
    without any scenario structure.
    """
    df = parse_jsonl_file(filepath)
    if df.empty:
        raise ValueError(f"No valid records in {filepath}")
    logger.info(f"Loaded {len(df)} records from {filepath.name}")
    return df

In [ ]:
def clean_logs(df: pd.DataFrame, warmup_seconds: int = WARMUP_SECONDS) -> pd.DataFrame:
    """
    Clean raw log DataFrame:
      1. Drop error rows
      2. Keep only rows that have all required columns
      3. Parse timestamp → datetime
      4. Cast metric columns to float
      5. Sort by (node, timestamp)
      6. Remove exact duplicates
      7. Remove per-node warmup period

    Args:
        df             : raw DataFrame from load_raw_logs()
        warmup_seconds : seconds to drop from the start of each node's timeline

    Returns:
        Cleaned DataFrame ready for feature engineering.
    """
    n_raw = len(df)

    # 1. Drop error rows and remove the error column entirely
    if "error" in df.columns:
        mask = df["error"].notna()
        logger.info(f"  Error rows dropped   : {mask.sum():,}")
        df = df[~mask].drop(columns=["error"])

    # 2. Require 'node' column — should always exist after load_raw_logs
    if "node" not in df.columns:
        raise ValueError(
            "'node' column is missing. Make sure load_raw_logs() was used "
            "or that the JSONL file contains a 'node' field."
        )
    df = df.dropna(subset=["node"])

    # Keep only rows that have the required metric columns present
    available = [c for c in REQUIRED_RAW_COLS if c in df.columns]
    missing   = [c for c in REQUIRED_RAW_COLS if c not in df.columns]
    if missing:
        logger.warning(f"  Missing columns: {missing}")
    metric_only = [c for c in available if c not in ("timestamp", "node")]
    df = df.dropna(subset=metric_only)

    # 3. Parse timestamp
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    # 4. Cast metrics to float
    for col in metric_only:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=metric_only)

    # 5. Sort
    df = df.sort_values(["node", "timestamp"]).reset_index(drop=True)

    # 6. Remove exact duplicates (same node + timestamp)
    before_dedup = len(df)
    df = df.drop_duplicates(subset=["node", "timestamp"])
    logger.info(f"  Duplicates removed   : {before_dedup - len(df):,}")

    # 7. Warmup removal — drop first `warmup_seconds` per node
    #    Use transform("min") instead of groupby+apply to avoid pandas 2.x
    #    dropping the groupby key column from the result.
    if warmup_seconds > 0:
        before_warmup = len(df)
        node_start = df.groupby("node")["timestamp"].transform("min")
        elapsed = (df["timestamp"] - node_start).dt.total_seconds()
        df = df[elapsed >= warmup_seconds].reset_index(drop=True)
        logger.info(f"  Warmup rows removed  : {before_warmup - len(df):,}")

    logger.info(f"  Clean records        : {len(df):,}  (from {n_raw:,} raw)")
    logger.info(f"  Nodes present        : {sorted(df['node'].unique())}")
    return df

In [ ]:
def _safe_ratio(num: float, den: float, fill: float = 0.0) -> float:
    """Scalar safe division."""
    return num / den if den > 0 else fill


def engineer_features_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute features for a full DataFrame sorted by (node, timestamp).

    Rate features   : delta(metric) / delta(seconds)
    Ratio features  : delta(numerator) / delta(denominator)   ← delta-based fix
    Absolute features: instantaneous value from current row

    The first row of each node is dropped (no previous row to diff against).
    Rate columns are then clipped to the 1st–99th percentile of this dataset.

    Returns DataFrame with columns: timestamp, node, <FEATURE_COLS>, ...
    """
    feat_frames = []

    for node, grp in tqdm(df.groupby("node"), desc="Feature engineering", leave=False):
        grp = grp.sort_values("timestamp").copy()

        # Time delta in seconds between consecutive measurements
        dt = grp["timestamp"].diff().dt.total_seconds()
        dt = dt.clip(lower=0.1)   # avoid division by near-zero

        # ── Absolute features
        grp["pit_size"] = grp["nPitEntries"].astype(float)
        grp["cs_size"]  = grp["nCsEntries"].astype(float)

        # ── Rate features (delta / dt)
        grp["pit_growth_rate"]    = grp["nPitEntries"].diff()                            / dt
        grp["in_interests_rate"]  = grp["nInInterests"].diff()                           / dt
        grp["out_interests_rate"] = grp["nOutInterests"].diff()                          / dt
        grp["in_data_rate"]       = grp["nInData"].diff()                                / dt
        grp["nack_rate"]          = (grp["nInNacks"] + grp["nOutNacks"]).diff()          / dt

        # ── Ratio features — delta-based (THE FIX)
        #    Old code used cumulative totals → ratio drifted as nMisses grew.
        #    New code uses per-interval deltas → stable across all time windows.
        d_hits   = grp["nHits"].diff()
        d_misses = grp["nMisses"].diff()
        d_cache  = d_hits + d_misses
        grp["cache_hit_ratio"] = np.where(d_cache > 0, d_hits / d_cache, 0.0)

        d_sat   = grp["nSatisfiedInterests"].diff()
        d_unsat = grp["nUnsatisfiedInterests"].diff()
        d_total = d_sat + d_unsat
        grp["satisfaction_ratio"]  = np.where(d_total > 0, d_sat   / d_total, 0.0)
        grp["unsatisfied_ratio"]   = np.where(d_total > 0, d_unsat / d_total, 0.0)

        feat_frames.append(grp)

    feat_df = pd.concat(feat_frames, ignore_index=True)

    # Keep only the needed columns (+ optional provenance)
    provenance = [c for c in ["_scenario", "source_scenarios", "elapsed_sec"] if c in feat_df.columns]
    keep = ["timestamp", "node"] + FEATURE_COLS + provenance
    feat_df = feat_df[[c for c in keep if c in feat_df.columns]].copy()

    # Drop first rows per node (NaN from .diff()) and any remaining NaN
    n_before = len(feat_df)
    feat_df = feat_df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
    logger.info(f"  Feature rows after diff-drop: {len(feat_df):,}  (dropped {n_before - len(feat_df):,})")

    # Clip rate features to 1st–99th percentile to suppress outliers
    clip_bounds = {}
    for col in RATE_COLS:
        lo = float(feat_df[col].quantile(CLIP_QUANTILE_LO))
        hi = float(feat_df[col].quantile(CLIP_QUANTILE_HI))
        feat_df[col] = feat_df[col].clip(lo, hi)
        clip_bounds[col] = {"lo": lo, "hi": hi}

    # Attach clip bounds as a DataFrame attribute so they can be saved
    feat_df.attrs["clip_bounds"] = clip_bounds
    return feat_df


def engineer_features_pair(prev: dict, curr: dict,
                           clip_bounds: Optional[dict] = None
                           ) -> Tuple[Optional[np.ndarray], Optional[Dict]]:
    """
    Compute feature vector from two consecutive raw log entries.
    Mirrors engineer_features_df() exactly.

    Args:
        prev        : previous raw log dict
        curr        : current raw log dict
        clip_bounds : dict loaded from clip_bounds.json; if provided, rate
                      features are clipped to training-time bounds before
                      the model sees them.

    Returns:
        (feature_array, feature_dict) or (None, None) on error.
    """
    try:
        # Time delta
        prev_ts = datetime.fromisoformat(prev["timestamp"].replace("Z", "").split(".")[0])
        curr_ts = datetime.fromisoformat(curr["timestamp"].replace("Z", "").split(".")[0])
        dt = (curr_ts - prev_ts).total_seconds()
        if dt <= 0:
            return None, None
        dt = max(dt, 0.1)

        # ── Absolute
        pit_size = float(curr.get("nPitEntries", 0))
        cs_size  = float(curr.get("nCsEntries",  0))

        # ── Rates
        pit_growth_rate    = (float(curr.get("nPitEntries",  0)) - float(prev.get("nPitEntries",  0))) / dt
        in_interests_rate  = (float(curr.get("nInInterests", 0)) - float(prev.get("nInInterests", 0))) / dt
        out_interests_rate = (float(curr.get("nOutInterests",0)) - float(prev.get("nOutInterests",0))) / dt
        in_data_rate       = (float(curr.get("nInData",      0)) - float(prev.get("nInData",      0))) / dt
        nack_diff  = (float(curr.get("nInNacks",  0)) + float(curr.get("nOutNacks",  0))) \
                   - (float(prev.get("nInNacks",  0)) + float(prev.get("nOutNacks",  0)))
        nack_rate  = nack_diff / dt

        # ── Ratios — delta-based (matches engineer_features_df)
        d_hits   = float(curr.get("nHits",   0)) - float(prev.get("nHits",   0))
        d_misses = float(curr.get("nMisses", 0)) - float(prev.get("nMisses", 0))
        d_cache  = d_hits + d_misses
        cache_hit_ratio = d_hits / d_cache if d_cache > 0 else 0.0

        d_sat    = float(curr.get("nSatisfiedInterests",   0)) - float(prev.get("nSatisfiedInterests",   0))
        d_unsat  = float(curr.get("nUnsatisfiedInterests", 0)) - float(prev.get("nUnsatisfiedInterests", 0))
        d_total  = d_sat + d_unsat
        satisfaction_ratio = d_sat  / d_total if d_total > 0 else 0.0
        unsatisfied_ratio  = d_unsat / d_total if d_total > 0 else 0.0

        fvec = [
            pit_size, pit_growth_rate, cs_size,
            cache_hit_ratio, satisfaction_ratio, unsatisfied_ratio,
            in_interests_rate, out_interests_rate, in_data_rate, nack_rate,
        ]

        # Apply saved clip bounds so inference matches training pre-processing
        if clip_bounds:
            for i, col in enumerate(FEATURE_COLS):
                if col in clip_bounds:
                    fvec[i] = max(clip_bounds[col]["lo"], min(clip_bounds[col]["hi"], fvec[i]))

        fdict = dict(zip(FEATURE_COLS, fvec))
        return np.array(fvec), fdict

    except Exception as e:
        logger.error(f"Feature computation error: {e}")
        return None, None

In [ ]:
def prepare_dataset(
    scenarios: List[str],
    label: str = "normal",
    log_root: Path = LOG_ROOT,
    warmup_seconds: int = WARMUP_SECONDS,
    save_path: Optional[Path] = None,
) -> pd.DataFrame:
    """
    End-to-end dataset preparation for one set of scenarios.

    Args:
        scenarios     : scenario folder names
        label         : descriptive label ('normal', 'ifa_attack', 'cp_attack' …)
        log_root      : root log directory
        warmup_seconds: warmup period to drop
        save_path     : if given, saves CSV here

    Returns:
        Feature DataFrame ready for training / evaluation.
    """
    logger.info(f"\n{'='*60}")
    logger.info(f"Preparing dataset: {label}")
    logger.info(f"Scenarios: {scenarios}")

    raw   = load_raw_logs(scenarios, log_root)
    clean = clean_logs(raw, warmup_seconds)
    feats = engineer_features_df(clean)

    # Carry scenario label
    feats["source_scenarios"] = label

    logger.info(f"Final shape: {feats.shape}")
    logger.info(f"Nodes: {sorted(feats['node'].unique())}")
    logger.info(f"Clip bounds saved in feats.attrs['clip_bounds']")

    if save_path:
        feats.drop(columns=["_scenario"], errors="ignore").to_csv(save_path, index=False)
        logger.info(f"Saved → {save_path}")

    return feats

In [ ]:
def explore_features(feat_df: pd.DataFrame, title: str = "Dataset") -> pd.DataFrame:
    """
    Print descriptive statistics and show distribution plots.
    Returns the stats DataFrame for inspection.
    """
    stats = feat_df[FEATURE_COLS].describe().T.round(4)
    print(f"\n{'='*60}")
    print(f"Feature statistics — {title}")
    print(f"{'='*60}")
    print(stats.to_string())

    # Distribution drift check against training scaler (if loaded)
    if MODEL_PATH.exists():
        try:
            pipe = joblib.load(MODEL_PATH)
            scaler = pipe.named_steps["scaler"]
            print(f"\n  Drift from training distribution (z = |μ_infer - μ_train| / σ_train):")
            print(f"  {'Feature':<22}  {'infer_μ':>10}  {'train_μ':>10}  {'train_σ':>10}  {'z':>8}  status")
            print(f"  {'─'*22}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*8}  {'─'*10}")
            for j, col in enumerate(FEATURE_COLS):
                mu_i = feat_df[col].mean()
                mu_t = scaler.mean_[j]
                si_t = scaler.scale_[j]
                z    = abs(mu_i - mu_t) / (si_t + 1e-12)
                st   = "<< DRIFTED" if z > 3 else ("slight" if z > 1.5 else "ok")
                print(f"  {col:<22}  {mu_i:>10.4f}  {mu_t:>10.4f}  {si_t:>10.4f}  {z:>8.3f}  {st}")
        except Exception:
            pass

    try:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(2, 5, figsize=(18, 7))
        axes = axes.flatten()
        for i, col in enumerate(FEATURE_COLS):
            axes[i].hist(feat_df[col].dropna(), bins=60, color="steelblue", alpha=0.75, edgecolor="none")
            axes[i].set_title(col, fontsize=9, fontweight="bold")
            axes[i].axvline(feat_df[col].mean(), color="red",    linestyle="--", linewidth=1.2, label="mean")
            axes[i].axvline(feat_df[col].median(), color="green", linestyle=":",  linewidth=1.2, label="median")
            axes[i].set_yticks([])
        axes[0].legend(fontsize=7)
        fig.suptitle(f"Feature distributions — {title}", fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.show()
    except ImportError:
        logger.info("matplotlib not available — skipping plots")

    return stats

In [ ]:
def train_model(
    feat_df: pd.DataFrame,
    n_estimators: int  = IF_N_ESTIMATORS,
    contamination: float = IF_CONTAMINATION,
    anomaly_percentile: int = ANOMALY_PERCENTILE,
) -> Tuple[Pipeline, float, np.ndarray]:
    """
    Train IsolationForest pipeline and compute anomaly threshold.

    Args:
        feat_df            : feature DataFrame (normal traffic only)
        n_estimators       : number of trees
        contamination      : expected fraction of anomalies in training data
        anomaly_percentile : bottom-N% of training scores → threshold

    Returns:
        (pipeline, threshold, train_scores)
    """
    X = feat_df[FEATURE_COLS].values
    logger.info(f"Training matrix shape: {X.shape}")

    pipeline = Pipeline([
        ("scaler",  StandardScaler()),
        ("iforest", IsolationForest(
            n_estimators  = n_estimators,
            contamination = contamination,
            max_samples   = "auto",
            random_state  = IF_RANDOM_STATE,
            n_jobs        = -1,
        )),
    ])

    logger.info(f"Training Isolation Forest (n_estimators={n_estimators}, "
                f"contamination={contamination}) …")
    pipeline.fit(X)
    logger.info("Training complete.")

    # Scores on training data: higher = more normal
    train_scores = pipeline.decision_function(X)

    # Threshold = bottom-N% of training scores
    threshold = float(np.percentile(train_scores, anomaly_percentile))
    logger.info(f"Threshold (p{anomaly_percentile} of training scores): {threshold:.6e}")

    return pipeline, threshold, train_scores

In [ ]:
def analyse_threshold(train_scores: np.ndarray, threshold: float) -> None:
    """
    Print score distribution and flag-rate at several candidate thresholds.
    """
    print(f"\n{'='*60}")
    print("Threshold Analysis — training score distribution")
    print(f"{'='*60}")
    print(f"  Scores: min={train_scores.min():.6f}  max={train_scores.max():.6f}"
          f"  mean={train_scores.mean():.6f}  std={train_scores.std():.6f}")

    print(f"\n  Percentile → threshold → flag-rate on training data")
    print(f"  {'%ile':>6}  {'threshold':>14}  {'flag-rate':>10}")
    print(f"  {'─'*6}  {'─'*14}  {'─'*10}")
    for pct in [0.1, 0.5, 1, 2, 5]:
        thr  = np.percentile(train_scores, pct)
        rate = (train_scores < thr).mean() * 100
        marker = "  ← current" if abs(thr - threshold) < 1e-10 else ""
        print(f"  {pct:>6.1f}  {thr:>14.6e}  {rate:>9.2f}%{marker}")

    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(10, 4))
        plt.hist(train_scores, bins=80, color="steelblue", alpha=0.8, edgecolor="none")
        plt.axvline(threshold, color="red", linewidth=2, linestyle="--",
                    label=f"threshold = {threshold:.3e}")
        plt.xlabel("decision_function score", fontweight="bold")
        plt.ylabel("count", fontweight="bold")
        plt.title("Training score distribution", fontweight="bold")
        plt.legend()
        plt.tight_layout()
        plt.show()
    except ImportError:
        pass

In [ ]:
def save_artifacts(
    pipeline: Pipeline,
    threshold: float,
    feat_df: pd.DataFrame,
    clip_bounds: dict,
    train_scores: Optional[np.ndarray] = None,
) -> None:
    """
    Save all artifacts required for reproducible inference.
    """
    # 1. Pipeline
    joblib.dump(pipeline, MODEL_PATH)
    logger.info(f"Pipeline saved → {MODEL_PATH}")

    # 2. Threshold metadata + training score distribution
    #    score_min / score_max are used by _normalize_score() in the detector
    #    so the 0-100 display scale is calibrated to this model's actual range.
    meta = {
        "threshold"           : threshold,
        "contamination"       : IF_CONTAMINATION,
        "anomaly_percentile"  : ANOMALY_PERCENTILE,
        "n_estimators"        : IF_N_ESTIMATORS,
        "trained_on_nodes"    : sorted(feat_df["node"].unique().tolist()),
        "n_training_samples"  : int(len(feat_df)),
        "trained_at"          : datetime.utcnow().isoformat(),
    }
    if train_scores is not None:
        meta["score_min"]  = float(train_scores.min())
        meta["score_max"]  = float(train_scores.max())
        meta["score_mean"] = float(train_scores.mean())
        meta["score_std"]  = float(train_scores.std())
    THRESHOLD_PATH.write_text(json.dumps(meta, indent=2))
    logger.info(f"Threshold saved  → {THRESHOLD_PATH}")

    # 3. Feature column order
    FEATURES_PATH.write_text(json.dumps(FEATURE_COLS, indent=2))
    logger.info(f"Feature cols saved → {FEATURES_PATH}")

    # 4. Clip bounds  ← critical for inference parity
    CLIP_BOUNDS_PATH.write_text(json.dumps(clip_bounds, indent=2))
    logger.info(f"Clip bounds saved  → {CLIP_BOUNDS_PATH}")


def load_artifacts() -> Tuple[Pipeline, float, dict]:
    """
    Load pipeline, threshold, and clip bounds from MODEL_DIR.

    Returns:
        (pipeline, threshold, clip_bounds)
    """
    for p in [MODEL_PATH, THRESHOLD_PATH, FEATURES_PATH, CLIP_BOUNDS_PATH]:
        if not p.exists():
            raise FileNotFoundError(f"Missing artifact: {p}")

    pipeline    = joblib.load(MODEL_PATH)
    threshold   = json.loads(THRESHOLD_PATH.read_text())["threshold"]
    clip_bounds = json.loads(CLIP_BOUNDS_PATH.read_text())

    logger.info(f"Artifacts loaded from {MODEL_DIR}")
    logger.info(f"Threshold : {threshold:.6e}")
    return pipeline, threshold, clip_bounds


In [ ]:
class NDNAnomalyDetector:
    """
    Real-time Isolation Forest anomaly detector for NDN networks.

    Usage:
        detector = NDNAnomalyDetector()          # loads from MODEL_DIR
        result   = detector.ingest(raw_entry)    # call for every new log line
    """

    def __init__(self, model_dir: Path = MODEL_DIR):
        self.pipeline, self.threshold, self.clip_bounds = load_artifacts()
        self._buffers: Dict[str, deque] = {}   # node → deque(maxlen=2)
        logger.info("NDNAnomalyDetector ready")

    def ingest(self, raw_entry: dict) -> dict:
        """
        Process one raw log entry and return detection result.

        Returns dict with keys:
            node, timestamp, status, anomaly_score, normalized_score,
            is_anomaly, features, alert
        """
        node = raw_entry.get("node", "unknown")
        ts   = raw_entry.get("timestamp")

        if node not in self._buffers:
            self._buffers[node] = deque(maxlen=2)

        # Need at least one previous entry to compute deltas
        if len(self._buffers[node]) < 1:
            self._buffers[node].append(raw_entry)
            return self._result(node, ts, "buffering",
                                alert="Warming up — need one more measurement")

        prev  = self._buffers[node][-1]
        fvec, fdict = engineer_features_pair(prev, raw_entry, self.clip_bounds)

        if fvec is None:
            self._buffers[node].append(raw_entry)
            return self._result(node, ts, "skipped",
                                alert="Could not compute features")

        X     = fvec.reshape(1, -1)
        score = float(self.pipeline.decision_function(X)[0])
        norm  = self._normalize(score)
        is_anomaly = score < self.threshold

        self._buffers[node].append(raw_entry)

        alert = f"Score: {score:.6f}  ({norm:.1f}%)"
        if is_anomaly:
            alert += " ← ANOMALY"

        return self._result(node, ts, "scored",
                            anomaly_score=score, normalized_score=norm,
                            is_anomaly=is_anomaly, features=fdict, alert=alert)

    # ── helpers ───────────────────────────────────────────────────────────────

    @staticmethod
    def _result(node, ts, status, anomaly_score=None, normalized_score=None,
                is_anomaly=None, features=None, alert="") -> dict:
        return {
            "node"            : node,
            "timestamp"       : ts,
            "status"          : status,
            "anomaly_score"   : anomaly_score,
            "normalized_score": normalized_score,
            "is_anomaly"      : is_anomaly,
            "features"        : features,
            "alert"           : alert,
        }

    @staticmethod
    def _normalize(score: float) -> float:
        """Map score to 0–100 (100 = normal, 0 = most anomalous)."""
        norm = ((score + 0.5) / 2.5) * 100
        return max(0.0, min(100.0, norm))

    def reset_node(self, node: str) -> None:
        """Clear buffer for a node (e.g. on reconnection)."""
        self._buffers.pop(node, None)

    def reset_all(self) -> None:
        """Clear all per-node buffers."""
        self._buffers.clear()

In [ ]:
def batch_infer_file(
    jsonl_path: Path,
    detector: Optional[NDNAnomalyDetector] = None,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Run inference on every entry in a JSONL file.

    Args:
        jsonl_path : path to *.jsonl file
        detector   : NDNAnomalyDetector instance (created if None)
        verbose    : print per-row summary

    Returns:
        DataFrame with one row per scored entry.
    """
    if detector is None:
        detector = NDNAnomalyDetector()

    entries = []
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if line:
                entries.append(json.loads(line))

    logger.info(f"Scoring {len(entries)} entries from {jsonl_path.name}")

    rows = []
    for i, entry in enumerate(entries):
        result = detector.ingest(entry)
        result["row_idx"] = i
        rows.append(result)

        if verbose and result["status"] == "scored":
            flag = "  *** ANOMALY ***" if result["is_anomaly"] else ""
            print(f"  [{i:>3}] {result['timestamp']}  "
                  f"score={result['anomaly_score']:+.6f}  "
                  f"anomaly={result['is_anomaly']}{flag}")

    df = pd.DataFrame(rows)
    scored = df[df["status"] == "scored"]

    if len(scored):
        n_anom = scored["is_anomaly"].sum()
        print(f"\n  Scored: {len(scored)}   Anomalies: {n_anom} "
              f"({100*n_anom/len(scored):.1f}%)")
        print(f"  Score range: [{scored['anomaly_score'].min():+.6f}, "
              f"{scored['anomaly_score'].max():+.6f}]")

    return df

In [ ]:

def check_drift(
    infer_df: Optional[pd.DataFrame]     = None,
    infer_jsonl: Optional[Path]          = None,
    pipeline: Optional[Pipeline]         = None,
    clip_bounds: Optional[dict]          = None,
) -> pd.DataFrame:
    """
    Report per-feature z-score drift between an inference batch and the
    training distribution captured in the StandardScaler.

    Supply either:
        infer_df    — a feature DataFrame (columns = FEATURE_COLS)
        infer_jsonl — a raw JSONL file (features will be computed on the fly)

    Returns drift DataFrame.
    """
    # Load pipeline if not provided
    if pipeline is None:
        pipeline, _, clip_bounds_loaded = load_artifacts()
        if clip_bounds is None:
            clip_bounds = clip_bounds_loaded

    scaler = pipeline.named_steps["scaler"]

    # Build feature matrix from JSONL if needed
    if infer_df is None and infer_jsonl is not None:
        raw_df = load_single_jsonl(infer_jsonl)
        # No warmup removal for a short snippet — just clean & engineer
        raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"], errors="coerce")
        raw_df = raw_df.dropna(subset=["timestamp"])
        raw_df = raw_df.sort_values(["node", "timestamp"]).reset_index(drop=True)
        infer_df = engineer_features_df(raw_df)
        # Apply saved clip bounds manually (matches inference engine)
        if clip_bounds:
            for col in RATE_COLS:
                if col in clip_bounds:
                    infer_df[col] = infer_df[col].clip(
                        clip_bounds[col]["lo"], clip_bounds[col]["hi"])

    if infer_df is None or infer_df.empty:
        raise ValueError("Provide either infer_df or infer_jsonl")

    rows = []
    for j, col in enumerate(FEATURE_COLS):
        mu_i = infer_df[col].mean()
        mu_t = scaler.mean_[j]
        si_t = scaler.scale_[j]
        z    = abs(mu_i - mu_t) / (si_t + 1e-12)
        rows.append({
            "feature"     : col,
            "infer_mean"  : round(mu_i, 5),
            "train_mean"  : round(mu_t, 5),
            "train_std"   : round(si_t, 5),
            "z_score"     : round(z, 3),
            "status"      : "DRIFTED" if z > 3 else ("slight" if z > 1.5 else "ok"),
        })

    drift_df = pd.DataFrame(rows)
    print(f"\n{'='*70}")
    print("Distribution Drift Report")
    print(f"{'='*70}")
    print(drift_df.to_string(index=False))
    drifted = drift_df[drift_df["status"] == "DRIFTED"]["feature"].tolist()
    if drifted:
        print(f"\n  *** Drifted features: {drifted}")
        print("  → These features have very different values vs training data.")
        print("  → Verify they are computed the same way in training and inference.")
    else:
        print("\n  All features within 3σ of training distribution.")
    return drift_df


In [ ]:
def evaluate(
    feat_df: pd.DataFrame,
    pipeline: Optional[Pipeline] = None,
    threshold: Optional[float]   = None,
    label_col: str               = "label",
) -> dict:
    """
    Score a labelled dataset and print a confusion-matrix style report.

    Args:
        feat_df   : feature DataFrame with a label column
        pipeline  : trained pipeline (loaded from disk if None)
        threshold : anomaly threshold (loaded from disk if None)
        label_col : name of the ground-truth column (0=normal, 1=anomaly)

    Returns:
        dict with precision, recall, f1, accuracy, n_flagged.
    """
    if pipeline is None or threshold is None:
        pipeline, threshold, _ = load_artifacts()

    X      = feat_df[FEATURE_COLS].values
    scores = pipeline.decision_function(X)
    preds  = (scores < threshold).astype(int)

    if label_col not in feat_df.columns:
        # No labels — just report flag rate
        n_flagged = int(preds.sum())
        print(f"\n  No '{label_col}' column found.")
        print(f"  Flagged {n_flagged}/{len(preds)} ({100*n_flagged/len(preds):.2f}%) as anomaly")
        return {"n_flagged": n_flagged, "total": len(preds),
                "flag_rate": n_flagged / len(preds)}

    y_true = feat_df[label_col].values
    tp = int(((preds == 1) & (y_true == 1)).sum())
    tn = int(((preds == 0) & (y_true == 0)).sum())
    fp = int(((preds == 1) & (y_true == 0)).sum())
    fn = int(((preds == 0) & (y_true == 1)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (tp + tn) / len(y_true)

    print(f"\n{'='*50}")
    print("Evaluation Results")
    print(f"{'='*50}")
    print(f"  Threshold  : {threshold:.6e}")
    print(f"  Samples    : {len(y_true):,}  (anomaly={int(y_true.sum())}, normal={int((y_true==0).sum())})")
    print(f"\n  Confusion matrix:")
    print(f"               Predicted N  Predicted A")
    print(f"  Actual N       {tn:>8}     {fp:>8}")
    print(f"  Actual A       {fn:>8}     {tp:>8}")
    print(f"\n  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  F1        : {f1:.4f}")
    print(f"  Accuracy  : {accuracy:.4f}")

    return dict(precision=precision, recall=recall, f1=f1, accuracy=accuracy,
                tp=tp, tn=tn, fp=fp, fn=fn)

In [ ]:
if __name__ == "__main__":

    print("\n" + "="*60)
    print("  NDN Pipeline — End-to-end Demo")
    print("="*60)

    # ── Step 1: Try to re-train from raw logs ─────────────────────────────────
    # If logs are available, build a fresh training dataset with the fixed
    # delta-based features and retrain.

    NORMAL_SCENARIOS  = ["logs1", "logs_mesh1", "logs_dumbbell1"]
    normal_csv = DATASET_DIR / "normal_traffic_features_fixed.csv"

    retrained = False
    if any((LOG_ROOT / s).exists() for s in NORMAL_SCENARIOS):
        print("\n[1/4] Building training dataset from raw logs …")
        normal_df = prepare_dataset(
            scenarios     = [s for s in NORMAL_SCENARIOS if (LOG_ROOT / s).exists()],
            label         = "normal",
            save_path     = normal_csv,
        )
        clip_bounds = normal_df.attrs["clip_bounds"]

        print("\n[2/4] Exploring training features …")
        explore_features(normal_df, title="Normal Traffic (fixed features)")

        print("\n[3/4] Training model …")
        pipeline, threshold, train_scores = train_model(normal_df)
        analyse_threshold(train_scores, threshold)

        print("\n[4/4] Saving artifacts …")
        save_artifacts(pipeline, threshold, normal_df, clip_bounds, train_scores)
        retrained = True
    else:
        print("\n[1/4] Raw logs not found — skipping training, loading saved model.")

    # ── Step 2: Inference on c1_after_600.jsonl ───────────────────────────────
    snippet = ROOT_DIR / "c1_after_600.jsonl"
    if snippet.exists():
        print(f"\n{'='*60}")
        print(f"  Batch inference on {snippet.name}")
        print(f"{'='*60}")

        detector = NDNAnomalyDetector()
        results  = batch_infer_file(snippet, detector=detector, verbose=True)

        # ── Step 3: Drift check ───────────────────────────────────────────────
        print(f"\n{'='*60}")
        print("  Drift check vs training distribution")
        print(f"{'='*60}")
        check_drift(infer_jsonl=snippet)

    else:
        print(f"\nSnippet {snippet} not found — skipping inference demo.")

    print("\nDone.")